# Langfuse 01 · 追踪、会话与提示词

Agent 上线后是个黑盒：它为什么调了这个工具？这一轮花了多少钱、多少 token？
**Langfuse** 是开源的 LLM 应用可观测平台，和 LangChain 官方的 LangSmith 做的是同一件事
（调试 / 监控 / 评估），差别是 LangSmith 收费、不开源，Langfuse 开源、可自建。

它的四大能力，本课讲前三样，第四样留给下一课：

| 能力 | 做什么 | 本课位置 |
|---|---|---|
| Tracing 追踪 | 记录每次模型 / 工具调用的输入、输出、token、耗时、成本，串成一棵调用树 | 第 1 节 |
| Session / User 会话与用户 | 给 trace 打 `session_id` / `user_id`，把散落的调用聚成「会话」「用户」两个视图 | 第 2 节 |
| Prompt Management 提示词托管 | 提示词不进代码，放控制台改，带版本与 `label` 灰度 | 第 3 节 |
| Evaluation 评估 | 回答好不好（打分 / RAG / Agent 指标） | 下一课 `02_评估与打分.ipynb` |

> 本 notebook 由 `Agent/06_langfuse/` 下 6 个脚本合并而成：
> `01_追踪.py`（原版 73 行）+ `01_追踪_jxsd.py`（完整版 439 行）、
> `02_会话和用户.py`（原版 54 行）+ `02_会话和用户_jxsd.py`（完整版 239 行）、
> `03_提示词管理.py`（原版 63 行）+ `03_提示词管理_jxsd.py`（完整版 264 行）。

**官方文档**
- 可观测性总览：<https://langfuse.com/docs/observability/overview>

## 运行条件

| 项 | 说明 |
|---|---|
| 🔴 运行档位 | **需外部服务** —— 真实调用 `.env` 里配置的大模型，并连本机 Langfuse（`http://localhost:3001`） |
| 依赖 | `langchain` / `deepagents` / `langfuse`（本项目 venv 已装） |
| 密钥 | `settings.api_key`（模型）、`settings.langfuse_public_key` / `langfuse_secret_key`（Langfuse，已配置） |
| 前置服务 | 本机 Langfuse 服务端（docker 自建，端口 3001，v4 的 events_only 模式） |
| 预计耗时 | 约 30~60 秒（真实模型调用 10 余次） |

> 🔴 的含义：这个 notebook **不需要你另开一个窗口起服务**，但依赖「本机已经跑着一个
> Langfuse 服务端」。端点没起来时，本课照样能跑（走降级演示），只是追踪数据不会真的上传
> —— 见下方「前置条件自检」那一格。

## 本节地图

先看「埋点数据」从你的代码到 Langfuse 后台要经过哪几步，再看三条 trace 之间的关系。

```mermaid
graph LR
    A["你的 Agent 代码"] --> B["Langfuse SDK 埋点<br/>@observe / CallbackHandler / 手工 span"]
    B --> C["异步批量上报<br/>get_client().flush()"]
    C --> D["Langfuse 服务端<br/>localhost:3001"]
    D --> E["Tracing 追踪<br/>trace 树"]
    D --> F["Sessions / Users<br/>按会话、用户聚合"]
    D --> G["Prompts 提示词<br/>版本 + label"]
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 阶段 | 谁在干活 | 关键 API |
|---|---|---|
| 埋点 | Langfuse SDK | `@observe` / `CallbackHandler` / `start_as_current_observation` |
| 上报 | SDK 后台线程 | `get_client().flush()`（不 flush 默认批量异步刷） |
| 落库 | Langfuse 服务端 | `Tracing` / `Sessions` / `Users` / `Prompts` 四个页面 |

本课与上下节的衔接：上一章 `05_mcp` 讲「Agent 怎么调外部工具」，本课开始回答
「调完之后怎么看清它干了什么」；下一课 `02_评估与打分.ipynb` 则在本课的 trace 之上做评估。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

这一格是**前置条件自检**：先看密钥和端点是否就绪，再看模型配置。
它只读不写、只打印状态，不会因为缺东西而中断 —— 缺了就走后面各节的降级演示。

> 源文件里只有「密钥是否为空」一个开关（`LANGFUSE_READY`）；这里**额外**探一下端点是否
> 可达，纯粹是为了在跑之前把「密钥配了、但 docker 没起」这种最常见的卡点直接指出来。

In [ ]:
# ===== 前置条件自检：只打印状态，不中断 =====
from config import settings

_lf_key_ok = bool(settings.langfuse_public_key and settings.langfuse_secret_key)
print("Langfuse 密钥：", "已配置" if _lf_key_ok else "未配置（下方演示会走降级路径）")
print("Langfuse 端点：", settings.langfuse_host)

# 端点探测是本 notebook 新增的：源文件只判断「密钥是否为空」。
# 密钥配了但 docker 没起时，探测会失败，这里直接点出来。
import urllib.request
try:
    with urllib.request.urlopen(settings.langfuse_host + "/api/public/health", timeout=3) as _r:
        print("端点健康检查：", "OK" if _r.status == 200 else "HTTP " + str(_r.status))
except Exception as _e:      # noqa: BLE001 —— 探测失败不影响演示
    print("端点健康检查：不可达 →", type(_e).__name__, "（追踪数据无法上传，演示仍可继续）")

print("模型：", settings.model_name, "@", settings.base_url)

## 1. 追踪（Tracing）：看清每一次调用

课案开头的一句话把定位说清楚了：

> LangSmith 是 LangChain 官方提供的智能体调试、监控和评估平台，收费、不开源；
> Langfuse 是开源的、同样功能的平台，本节只讲 Langfuse。

Trace 到底记录了什么？这是本节要建立的直觉：

- 每次模型调用：输入 / 输出 / token 数 / 耗时 / 成本
- 每次工具调用：工具名 / 参数 / 返回值
- 以及这些步骤之间「谁包着谁」的调用链路树（trace tree）

课案给了三种埋点方式，本节逐一演示：

| 方式 | 适用对象 | 自动化程度 |
|---|---|---|
| `CallbackHandler` | LangChain / LangGraph / DeepAgents | 全自动 |
| `@observe` 装饰器 | 任意普通 Python 函数 | 一行 |
| `start_as_current_observation` | 需要手工控制 span 边界的代码 | 手工 |

### 1.1 课案原版：两种接入方式的最短实现

课案原版只有 73 行，正好演示两种接入：`@observe` 装饰器（任意函数）、
`CallbackHandler`（LangChain 全链路自动埋点）。**先看最短实现，再看完整版**，
两者的差距就是本节要补的「三种方式 + 降级演示」。

In [ ]:
# ---------- 课案原版：导入 + 客户端 ----------
from langchain.chat_models import init_chat_model
from langfuse import Langfuse, get_client, observe
from langfuse.langchain import CallbackHandler
from config import settings

lf = Langfuse(
    public_key=settings.langfuse_public_key,
    secret_key=settings.langfuse_secret_key,
    host=settings.langfuse_host,
)

方式一：`@observe` 装饰器。普通业务函数加个装饰器就有完整追踪，
每次调用会生成一条 trace（span）。

In [ ]:
@observe  # 这个函数会被自动追踪：每次调用生成一个 trace
def generate_answer(question: str) -> str:
    """普通业务函数，加个装饰器就有完整追踪"""
    llm = init_chat_model(
        model_provider="openai",
        model=settings.model_name,
        api_key=settings.api_key,
        base_url=settings.base_url,
    )
    return llm.invoke(question).content


def demo_observe():
    print("AI：", generate_answer("一句话解释什么是 Langfuse"))
    get_client().flush()  # 立刻上传（否则可能等批量刷新）

方式二：`CallbackHandler`。把它塞进 `invoke` 的 `config["callbacks"]`，
LangChain 整条链 / 工具 / 模型调用都会被自动埋点。

In [ ]:
def demo_callback():
    llm = init_chat_model(
        model_provider="openai",
        model=settings.model_name,
        api_key=settings.api_key,
        base_url=settings.base_url,
    )
    handler = CallbackHandler()  # LangChain 全链路自动埋点
    response = llm.invoke("什么是 RAG？一句话回答。", config={"callbacks": [handler]})
    print("AI：", response.content)
    get_client().flush()  # 立即上传（v4 用 get_client().flush()）

In [ ]:
# ---------- 课案原版：跑两种方式 ----------
demo_observe()
demo_callback()
print("已上传到 Langfuse，去控制台查看：", settings.langfuse_host)

### 预期输出

```text
AI： Langfuse 是一个开源的 LLM 可观测性平台，用于追踪、监控和评估大模型应用的每一次调用……
AI： RAG 是一种检索增强生成技术……
已上传到 Langfuse，去控制台查看： http://localhost:3001
```

> `AI：` 后面的内容是**真实大模型返回**，每次跑都会不同，上面只是示意。
> 稳定不变的是：两行 `AI：`（两次真实调用）+ 末尾那行「已上传到 … 控制台查看」。
> 打开 `http://localhost:3001` 的 Tracing 页，能看到刚生成的两条 trace。

### 1.2 完整版：三种埋点方式逐一展开

完整版 439 行，多做了三件事：

1. **客户端初始化加了开关**：密钥为空时干脆不实例化真客户端（否则 SDK 会往 stderr
   刷认证错误），并给出一套**降级演示** —— 用 LangChain 自带回调自己收集调用树打印出来，
   打印的字段就是「本应上报给 Langfuse 的数据」；
2. **三种埋点方式都演示**（原版只有前两种），第三种是给「非模型调用」圈耗时边界的手工 span；
3. 用 `create_deep_agent` 造一个带计算器工具的 Agent，让追踪树里出现真正的 tool 节点。

In [ ]:
# ---------- 完整版：导入 ----------
import json
import time
from contextlib import contextmanager
from types import SimpleNamespace

from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.tools import tool
from langfuse import Langfuse
from langfuse import get_client
from langfuse import observe as langfuse_observe
from langfuse.langchain import CallbackHandler
from config import settings

客户端初始化：先算一个开关 `LANGFUSE_READY`，空密钥时干脆不实例化真客户端
（SDK 拿不到凭据会往 stderr 刷 `Authentication error: ... Client will be disabled.`，
虽然不抛异常，但教学输出会很脏）。

In [ ]:
# ---------- 0. 客户端初始化：先判断密钥是否就绪 ----------
LANGFUSE_READY = bool(settings.langfuse_public_key and settings.langfuse_secret_key)

if LANGFUSE_READY:
    langfuse = Langfuse(
        public_key=settings.langfuse_public_key,
        secret_key=settings.langfuse_secret_key,
        host=settings.langfuse_host,          # 云端 https://cloud.langfuse.com；自建 http://localhost:3000
    )
else:
    langfuse = None                            # 降级演示不需要客户端

# 项目统一用 init_chat_model 初始化大模型（课案原文是 ChatOpenAI 直连，参数同样来自 settings）
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


def _observe_local(*dargs, **_dkwargs):
    """降级版 @observe：不做任何埋点，仅把原函数原样返回。

    保留这个壳子，是为了让下面的业务函数**保持课案里的代码形状**
    （函数上方照样写着 @observe），学员对照课案时不会觉得写法变了。
    """
    def _deco(func):
        return func

    return dargs[0] if (dargs and callable(dargs[0])) else _deco


# 密钥就绪 → 用 Langfuse 官方装饰器；否则 → 本地空壳
observe = langfuse_observe if LANGFUSE_READY else _observe_local

#### 1.2.1 本地「降级版 Langfuse」：自己收集调用树

下面这个 `LocalTraceCollector` 顶替的就是 Langfuse 的 `CallbackHandler`。
两者监听的是**同一批 LangChain 回调事件**，区别只是：Langfuse 把事件转成 span 上传，
这里只是攒在内存里打印 —— 所以打印出来的字段，就是「本应上报的数据」。

| LangChain 回调 | Langfuse observation 类型 |
|---|---|
| `on_chain_*` | span / chain |
| `on_tool_*` | tool |
| `on_llm_*` | generation（带 token 与成本） |

In [ ]:
# ---------- 1. 本地「降级版 Langfuse」：自己收集调用树 ----------
class LocalTraceCollector(BaseCallbackHandler):
    """监听 LangChain 回调，把整条调用链收集成一棵树并打印出来。"""

    def __init__(self, trace_name: str):
        super().__init__()
        self.trace_name = trace_name
        self.nodes: dict[str, dict] = {}      # run_id → 节点
        self.roots: list[str] = []            # 没有父节点的根 run

    # ---- 内部工具：登记一个节点的开始 / 结束 ----
    def _begin(self, kind: str, name: str, run_id, parent_run_id, payload=None):
        key = str(run_id)
        parent = str(parent_run_id) if parent_run_id else None
        # 登记节点时顺手记开始时间，_end 里再算耗时 —— 这就是 Langfuse 算 latency 的做法。
        self.nodes[key] = {
            "kind": kind, "name": name, "parent": parent,
            "input": payload, "output": None, "elapsed": None,
            "t0": time.perf_counter(), "children": [],
        }
        if parent and parent in self.nodes:
            self.nodes[parent]["children"].append(key)   # 挂到父节点下面，形成树
        else:
            self.roots.append(key)                       # 否则它就是一条 trace 的根

    def _end(self, run_id, output=None):
        node = self.nodes.get(str(run_id))
        # 结束时补上输出与耗时；找不到节点说明配对事件丢了，静默忽略即可。
        if node is not None:
            node["elapsed"] = time.perf_counter() - node["t0"]
            node["output"] = output

    @staticmethod
    def _node_name(serialized, metadata, kwargs, prefer_serialized=False) -> str:
        """给节点起个可读的名字。

        LangGraph / DeepAgents 里 serialized 常常是 None，真正有用的是
        kwargs['name']（图节点名）和 metadata['langgraph_node']（图节点名）；
        但工具的 serialized 里带着真正的工具名（calculator），要优先用它，
        否则树上只会看到 LangGraph 的节点名 tools。
        """
        meta = metadata or {}
        candidates = [(serialized or {}).get("name"), kwargs.get("name"), meta.get("langgraph_node")]
        if not prefer_serialized:
            candidates = [kwargs.get("name"), meta.get("langgraph_node"), (serialized or {}).get("name")]
        for candidate in candidates:
            if candidate:
                return str(candidate)
        return "chain"

    # ---- LangChain 回调钩子：链 / 工具 / 大模型 ----
    def on_chain_start(self, serialized, inputs, *, run_id, parent_run_id=None,
                       metadata=None, **kwargs):
        self._begin("chain", self._node_name(serialized, metadata, kwargs),
                    run_id, parent_run_id, payload=inputs)

    def on_chain_end(self, outputs, *, run_id, **kwargs):
        self._end(run_id, outputs)

    def on_tool_start(self, serialized, input_str, *, run_id, parent_run_id=None,
                      metadata=None, **kwargs):
        self._begin("tool", self._node_name(serialized, metadata, kwargs, prefer_serialized=True),
                    # 工具回调要多传 prefer_serialized=True —— 只有它的 serialized 里带真正的工具名。
                    run_id, parent_run_id, payload=input_str)

    def on_tool_end(self, output, *, run_id, **kwargs):
        self._end(run_id, output)

    def on_llm_start(self, serialized, prompts, *, run_id, parent_run_id=None,
                     metadata=None, **kwargs):
        self._begin("generation", self._node_name(serialized, metadata, kwargs),
                    run_id, parent_run_id, payload=prompts)

    def on_llm_end(self, response, *, run_id, **kwargs):
        # token 用量就藏在 LLMResult 里，Langfuse 用它算成本，这里也顺手取出来
        text, usage = None, None
        try:
            text = response.generations[0][0].text or response.generations[0][0].message.content
        except Exception:
            pass
        try:
            raw = response.llm_output.get("token_usage") or {}
            # 只留三个关键数字，明细（缓存命中/推理 token）太长，塞进一行会看不清
            usage = {k: raw.get(k) for k in ("prompt_tokens", "completion_tokens", "total_tokens")
                     if raw.get(k) is not None} or None
        except Exception:
            pass
        node = self.nodes.get(str(run_id))
        if node is not None:
            node["usage"] = usage
        self._end(run_id, text)

    # ---- 打印：等价于 Langfuse Web UI 里那棵 trace 树 ----
    def print_tree(self):
        print(f"trace  name={self.trace_name}")
        for key in self.roots:
            self._print_node(key, depth=1)

    # 递归打印：竖线和 ├─ 只是为了让终端里的层级一眼可读。
    def _print_node(self, key: str, depth: int):
        node = self.nodes[key]
        pad = "│  " * (depth - 1) + "├─ "
        elapsed = f"{node['elapsed']:.2f}s" if node["elapsed"] is not None else "-"
        line = f"{pad}[{node['kind']:<10}] {node['name']:<34} {elapsed:>7}"
        if node.get("usage"):
            line += "  tokens=" + "/".join(str(v) for v in node["usage"].values())
        # 递归打印子节点：深度 +1，缩进跟着加，终端里就能看出「谁包着谁」。
        print(line)
        if node["kind"] == "tool":
            print(f"{'│  ' * depth}   入参: {_short(node['input'])}")
            print(f"{'│  ' * depth}   出参: {_short(node['output'])}")
        for child in node["children"]:
            self._print_node(child, depth + 1)

    def as_ingestion_payload(self) -> dict:
        """把收集到的事件拼成「本应上传给 Langfuse 的报文」。"""
        observations = []
        # 逐个事件转成 observation —— 这正是 Langfuse ingestion API 期望的形状。
        for key, node in self.nodes.items():
            observations.append({
                "id": key,
                "traceId": "<由 SDK 生成的 trace_id>",
                "parentObservationId": node["parent"],
                "type": node["kind"].upper(),
                # 顶层是 trace + observations 两段，正是 Langfuse ingestion API 的请求体形状。
                "name": node["name"],
                "input": _short(node["input"], 120),
                "output": _short(node["output"], 120),
                "usage": node.get("usage"),
                "durationSeconds": round(node["elapsed"], 3) if node["elapsed"] else None,
            })
        return {"trace": {"name": self.trace_name}, "observations": observations}


def _short(value, limit: int = 60) -> str:
    """把任意对象压成一行短文本，方便塞进追踪节点里看。"""
    text = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False, default=str)
    text = " ".join(text.split())
    return text if len(text) <= limit else text[:limit] + "…"

手工 span：课案第三种埋点方式。密钥就绪时用官方 API（v4 里叫
`start_as_current_observation`），密钥缺失时退化成「本地计时 + 打印」。

In [ ]:
@contextmanager
def manual_span(name: str, **fields):
    """手工 span：课案第三种埋点方式。

    密钥就绪时用官方 API（v4 里叫 start_as_current_observation）：
        with langfuse.start_as_current_observation(name=name, as_type="span") as sp:
            ...业务代码...          # 期间发生的 LLM / 工具调用会自动挂到这个 span 下面
    密钥缺失时退化成「本地计时 + 打印」，保证脚本照样跑得下去。
    """
    if LANGFUSE_READY:
        with langfuse.start_as_current_observation(name=name, as_type="span", input=fields or None) as sp:
            yield sp
    else:
        t0 = time.perf_counter()
        holder = SimpleNamespace(update=lambda **kw: None)   # 空壳，保持调用形状一致
        print(f"  [降级] 进入手工 span：{name}  metadata={_short(fields)}")
        try:
            yield holder
        finally:
            print(f"  [降级] 退出手工 span：{name}  耗时 {time.perf_counter() - t0:.2f}s")

定义演示用的工具与 Agent。为了让追踪树里出现 tool 节点，这里挂一个计算器工具，
只提问算术题（`eval` 仅教学演示，生产要换成安全求值）。

In [ ]:
# ---------- 2. 定义演示用的工具与 Agent ----------
@tool
def calculator(expression: str) -> str:
    """执行数学计算。传入数学表达式字符串，如 '5*3+9'"""
    try:
        return str(eval(expression))       # 教学演示，表达式由模型生成，实际项目要换成安全求值
    except Exception:
        return "计算错误"


agent = create_deep_agent(model=llm, tools=[calculator])

#### 1.2.2 方式一：CallbackHandler（LangChain 生态一把梭）

给 `invoke` 的 `config` 挂一个回调处理器，整条链路自动埋点。密钥就绪时用真
`CallbackHandler`；否则用上面的 `LocalTraceCollector` 打印等价的 trace 树。

In [ ]:
# ---------- 3. 方式一：CallbackHandler（LangChain 生态一把梭） ----------
def demo_callback_handler():
    """课案主推方式：给 invoke 的 config 挂一个回调处理器，整条链路自动埋点。"""
    print("\n" + "=" * 72)
    print("方式一：CallbackHandler —— config={'callbacks': [langfuse_handler]}")
    print("=" * 72)

    if LANGFUSE_READY:
        # CallbackHandler 会自动把 LangChain 的每次链/工具/模型调用
        # 转成 Langfuse 的 observation，并串成一条 trace
        langfuse_handler = CallbackHandler()
        result = agent.invoke(
            {"messages": [{"role": "user", "content": "帮我算 5*3+9"}]},
            config={"callbacks": [langfuse_handler]},
        )
        print("AI：", result["messages"][-1].content)
        # last_trace_id 就是这次调用在 Langfuse 里的 trace id，后面打分要用它
        print("本次 trace_id：", langfuse_handler.last_trace_id)
        get_client().flush()      # 立刻上传（SDK 默认批量异步刷，不 flush 可能等几秒）
    else:
        collector = LocalTraceCollector("Langfuse 追踪演示")
        result = agent.invoke(
            {"messages": [{"role": "user", "content": "帮我算 5*3+9"}]},
            config={"callbacks": [collector]},
        )
        # 上面这行的形状和课案完全一致，只是把 langfuse_handler 换成了本地收集器
        print("AI：", result["messages"][-1].content)
        print("\n[降级] 下面是本次运行收集到的 trace 树（= Langfuse Web UI 里的样子）：")
        collector.print_tree()
        print("\n[降级] 本应上传给 Langfuse 的报文（节选）：")
        payload = collector.as_ingestion_payload()
        print(json.dumps(payload["observations"][:2], ensure_ascii=False, indent=2))

In [ ]:
demo_callback_handler()

### 预期输出

```text
========================================================================
方式一：CallbackHandler —— config={'callbacks': [langfuse_handler]}
========================================================================
AI： 5*3+9 = 24
本次 trace_id： <32 位十六进制>
```

> `AI：` 后面的计算结果和 `trace_id` 每次跑都会变；稳定的是标题分隔线、
> `AI：` 这一行、以及 `本次 trace_id：` 这一行。打开 Tracing 页能看到这条 trace 的
> `chain → agent → tools → generation` 调用树。

#### 1.2.3 方式二：@observe 装饰器（任意 Python 函数）

不限于 LangChain，任何 Python 函数加 `@observe` 就能追踪；函数内部再调被装饰的函数
会自动嵌套成父子 span。

In [ ]:
# ---------- 4. 方式二：@observe 装饰器（任意 Python 函数） ----------
@observe
def generate_answer(question: str) -> str:
    """普通业务函数，加个装饰器就有完整追踪。"""
    return llm.invoke(question).content


# 函数体只有一行，但 @observe 会为它生成 span；内部再调被装饰的函数会自动嵌套成父子。
def demo_observe():
    print("\n" + "=" * 72)
    print("方式二：@observe 装饰器 —— 不限于 LangChain，任何 Python 函数都能追踪")
    print("=" * 72)
    answer = generate_answer("一句话解释什么是 Langfuse")
    print("AI：", answer)

    if LANGFUSE_READY:
        # 当前 span 的 trace_id，函数内部随时可查
        print("当前 trace_id：", get_client().get_current_trace_id())
        get_client().flush()
    else:
        print("  [降级] @observe 变成了空壳（不上报），本次要上报的数据形如：")
        print(json.dumps({
            "type": "span",
            "name": "generate_answer",
            "input": {"question": "一句话解释什么是 Langfuse"},
            "output": answer,
            "metadata": {"sdk": "langfuse", "host": settings.langfuse_host},
        }, ensure_ascii=False, indent=2))

In [ ]:
demo_observe()

### 预期输出

```text
========================================================================
方式二：@observe 装饰器 —— 不限于 LangChain，任何 Python 函数都能追踪
========================================================================
AI： Langfuse 是一个开源的可观测性平台……
当前 trace_id： <32 位十六进制>
```

> 同上：`AI：` 内容与 `trace_id` 每次不同。`get_current_trace_id()` 在 `@observe`
> 装饰的函数内返回当前 span 的 trace id。

#### 1.2.4 方式三：手工 span（细粒度控制）

当一步操作**不是**「一次模型调用」时（比如检索、rerank、后处理），用 span 手工圈出
边界，Langfuse 才能算出这一步的耗时。下面第一个 span 里只有 `time.sleep(0.2)`（假装
向量检索），第二个 span 里是一次真的大模型调用。

In [ ]:
# ---------- 5. 方式三：手工 span（细粒度控制） ----------
def demo_manual_span():
    """当一步操作不是「一次模型调用」时（比如检索、rerank、后处理），
    用 span 手工圈出边界，Langfuse 才能算出这一步的耗时。"""
    print("\n" + "=" * 72)
    print("方式三：手工 span —— 给「非模型调用」的步骤也加上耗时统计")
    print("=" * 72)
    with manual_span("rag_retrieve", 检索库="教学知识库", top_k=3):
        time.sleep(0.2)                                   # 假装在做向量检索
        docs = ["Langfuse 是开源的 LLM 可观测性平台。", "它支持追踪、监控、评估、提示词管理。"]
    print("  检索到", len(docs), "条文档")
    with manual_span("answer_with_context", 文档数=len(docs)):
        answer = llm.invoke("根据资料一句话说明 Langfuse 的能力：\n" + "\n".join(docs))
    print("AI：", answer.content)
    if LANGFUSE_READY:
        get_client().flush()

In [ ]:
demo_manual_span()

### 预期输出

```text
========================================================================
方式三：手工 span —— 给「非模型调用」的步骤也加上耗时统计
========================================================================
  检索到 2 条文档
AI： Langfuse 支持追踪、监控、评估和提示词管理……
```

> 密钥就绪时这段会真把两个 span 上传；`AI：` 内容每次不同。手工 span 的意义：
> `rag_retrieve` 不是模型调用，不圈出来，看板上就查不到它的耗时瓶颈。

## 2. 会话（Session）与用户（User）

单条 trace 只能看到一次调用。要分析「某个用户的完整旅程」「某次会话的连续对话」，
需要给 trace 打上元数据标签。三个 ID 是三个不同粒度：

| 字段 | 代表什么 | 一次对话里有几条 | 后台能干什么 |
|---|---|---|---|
| `trace_id` | 一次请求 / 一次图执行 | 每轮一条 | 看这一轮的完整调用链 |
| `session_id` | 一次会话（一个聊天窗口） | 多条 trace | 把多轮对话连起来看上下文 |
| `user_id` | 一个用户（跨会话、跨天） | 跨会话累计 | 看某个用户的所有历史行为 |
| `tags` | 自定义标签（环境 / 版本 / 渠道） | 任意 | 按灰度版本、生产/测试筛选 |

### 2.1 课案原版：`configurable` 里的 session_id / user_id

LangChain 接入时，`CallbackHandler` 会自动从 `config["configurable"]` 里读
`session_id` / `user_id`，无需手动设置。模拟同一用户、同一会话里的两轮对话：

In [ ]:
# ---------- 课案原版：会话与用户 ----------
from langchain.chat_models import init_chat_model
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

In [ ]:
session_id = "session-20260909-001"
user_id = "user-1001"

# 模拟同一用户、同一会话里的两轮对话
handler = CallbackHandler()
config = {
    "callbacks": [handler],
    "configurable": {
        "session_id": session_id,  # 同一会话共享
        "user_id": user_id,        # 归属到用户
    },
}

r1 = llm.invoke("你好，我是小明", config=config)
r2 = llm.invoke("帮我写一句朋友圈文案，关于加班", config=config)

get_client().flush()
print("两轮对话已上传；在 Langfuse 里可按用户/会话筛选查看")

### 预期输出

```text
两轮对话已上传；在 Langfuse 里可按用户/会话筛选查看
```

> 两轮 `llm.invoke` 本身不打印（结果存进 `r1` / `r2`），所以稳定输出只有最后那一行。
> 在 Langfuse 后台的 Sessions 页，这两条 trace 会被串成一个会话（session_id 相同）。

### 2.2 完整版：两种写法 + 降级视图

完整版讲透三件事：

1. 课案给的 **metadata 写法** —— 把三个「魔法 key」塞进 `config["metadata"]`，
   Langfuse 回调会识别它们并提升为 trace 属性；
2. langfuse v4 的 **`propagate_attributes()` 上下文管理器** —— 一个 `with` 块内所有调用
   自动继承这些属性，不用逐个 invoke 传；
3. 降级时把「本应上报的 trace 元数据」打印出来，并模拟后台按 session / user 分组。

In [ ]:
# ---------- 完整版：会话与用户 ----------
import json
# from config import settings 是仓库根目录的统一配置，Langfuse 密钥与模型参数都从这里取。

from langchain.chat_models import init_chat_model
from langfuse import Langfuse
from langfuse import get_client
from langfuse import propagate_attributes
from langfuse.langchain import CallbackHandler
from config import settings

# ---------- 0. 客户端初始化：先判断密钥是否就绪 ----------
LANGFUSE_READY = bool(settings.langfuse_public_key and settings.langfuse_secret_key)

if LANGFUSE_READY:
    langfuse = Langfuse(
        public_key=settings.langfuse_public_key,
        secret_key=settings.langfuse_secret_key,
        host=settings.langfuse_host,
    )
else:
    langfuse = None      # 降级演示：不实例化真客户端，免得 SDK 往 stderr 刷认证错误

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

# 降级演示用：把「每条 trace 本应携带的元数据」攒起来，最后统一打印
LOCAL_TRACES: list[dict] = []


def _record(trace_name: str, session_id: str, user_id: str, tags: list[str],
            question: str, answer: str) -> None:
    """登记一条「本应上报的 trace」。真连上 Langfuse 后，这些字段就是后台的筛选项。"""
    LOCAL_TRACES.append({
        "id": f"trace-{len(LOCAL_TRACES) + 1:03d}",   # 真实环境里是 SDK 生成的 32 位 hex
        "name": trace_name,
        "sessionId": session_id,                      # 后台按「会话」筛选用它的值
        "userId": user_id,                            # 后台按「用户」筛选用它的值
        "tags": tags,                                 # 后台按标签二次筛选
        "input": question,
        "output": answer,
    })

#### 2.2.1 写法一（课案）：`config['metadata']` 里的三个魔法 key

In [ ]:
# ---------- 1. 课案写法：metadata 里的三个魔法 key ----------
def demo_metadata_keys():
    print("\n" + "=" * 72)
    print("写法一（课案）：config['metadata'] 里的 langfuse_session_id / user_id / tags")
    print("=" * 72)

    session_id = "session_001"
    user_id = "user_A"
    tags = ["production", "agent-v2"]

    turns = ["你好，我是小明", "帮我算一下 12*12"]   # 同一会话里的连续两轮
    for i, question in enumerate(turns, start=1):
        if LANGFUSE_READY:
            langfuse_handler = CallbackHandler()
            config = {
                "callbacks": [langfuse_handler],
                # last_trace_id 就是这一轮在 Langfuse 里的 trace id，后面打分要用它。
                "metadata": {
                    "langfuse_session_id": session_id,
                    "langfuse_user_id": user_id,
                    "langfuse_tags": tags,
                },
            }
            response = llm.invoke(question, config=config)
            answer = response.content
            print(f"  第{i}轮 trace_id = {langfuse_handler.last_trace_id}")
        else:
            # 降级：config 的形状与课案一模一样，只是回调换成了空壳
            answer = llm.invoke(question).content
            _record("chat", session_id, user_id, tags, question, answer)
        print(f"  第{i}轮  Q: {question}")
        print(f"          A: {answer[:60]}")

    # 多轮共用一个 session_id，后台就会把这两条 trace 串成一个会话。
    if LANGFUSE_READY:
        get_client().flush()

In [ ]:
demo_metadata_keys()

### 预期输出

```text
========================================================================
写法一（课案）：config['metadata'] 里的 langfuse_session_id / user_id / tags
========================================================================
  第1轮 trace_id = <32 位十六进制>
  第1轮  Q: 你好，我是小明
          A: 你好，小明！……
  第2轮 trace_id = <32 位十六进制>
  第2轮  Q: 帮我算一下 12*12
          A: 12×12=144……
```

> `A:` 是真实模型回答、`trace_id` 每次不同。两轮共用一个 `session_id`，
> 后台 Sessions 页会把它们串成一个会话。

#### 2.2.2 写法二（langfuse v4）：`propagate_attributes` 上下文管理器

In [ ]:
# ---------- 2. v4 写法：propagate_attributes 上下文管理器 ----------
def demo_propagate_attributes():
    """同一个会话换个用户再聊一句，用来对比「同会话 / 不同用户」在后台的差异。"""
    print("\n" + "=" * 72)
    print("写法二（langfuse v4）：with propagate_attributes(session_id=..., user_id=...)")
    print("=" * 72)

    session_id, user_id = "session_002", "user_B"
    question = "用一句话说明 session_id 和 user_id 的区别"

    if LANGFUSE_READY:
        # 这个 with 块里发生的**所有**调用都自动带上这些属性，
        # 不用再逐个 invoke 传 metadata，嵌套调用也会继承
        with propagate_attributes(
            session_id=session_id,
            user_id=user_id,
            tags=["staging", "v4-api"],
            trace_name="chat_propagated",
        ):
            langfuse_handler = CallbackHandler()
            answer = llm.invoke(question, config={"callbacks": [langfuse_handler]}).content
        print("  trace_id =", langfuse_handler.last_trace_id)
        get_client().flush()
    else:
        answer = llm.invoke(question).content
        _record("chat_propagated", session_id, user_id, ["staging", "v4-api"], question, answer)
# with 块里发生的所有调用都会继承这些属性，包括嵌套调用 —— 这是它比 metadata 省事的地方。

    print(f"  Q: {question}")
    print(f"  A: {answer[:60]}")

In [ ]:
demo_propagate_attributes()

### 预期输出

```text
========================================================================
写法二（langfuse v4）：with propagate_attributes(session_id=..., user_id=...)
========================================================================
  trace_id = <32 位十六进制>
  Q: 用一句话说明 session_id 和 user_id 的区别
  A: session_id 标识一次会话……
```

> 和写法一的差别只在「作用范围」：metadata 是**一次调用一份**，`propagate_attributes`
> 是**一个 with 块一份**（块内所有调用含嵌套自动继承）。

#### 2.2.3 降级视图：把「本应上报的元数据」打印出来

这一段只在密钥未就绪时才有输出；密钥就绪时它是空操作。它把 `LOCAL_TRACES` 里的记录
按 `session_id` / `user_id` 分组打印，模拟 Langfuse 后台 Sessions / Users 两个页面。

In [ ]:
# ---------- 3. 降级演示：把「本应上报的 trace 元数据」打印出来 ----------
def print_local_traces():
    if LANGFUSE_READY:
        return
    print("\n" + "=" * 72)
    print("[降级] 本应上报给 Langfuse 的 trace 记录（后台的「会话 / 用户」筛选就靠这些字段）")
    print("=" * 72)
    print(json.dumps(LOCAL_TRACES, ensure_ascii=False, indent=2))

    # 顺便按 session_id / user_id 分组，模拟 Langfuse 后台的两个视图
    # 这一步是「为什么要这两个 id」最直观的答案：同一批 trace，换一个 key 分组
    # 就得到两种完全不同的视图 —— 一个是「这次聊天说了什么」，一个是「这个人做过什么」。
    print("\n[降级] 换成 Langfuse 后台的视角：")
    by_session: dict[str, list[dict]] = {}
    by_user: dict[str, list[dict]] = {}
    for t in LOCAL_TRACES:
        # setdefault + append = 分组惯用法：key 不存在就建空列表，然后统一 append
        by_session.setdefault(t["sessionId"], []).append(t)
        by_user.setdefault(t["userId"], []).append(t)
    for sid, items in by_session.items():
        # 会话视图：只关心「这一窗里问了哪些问题」，顺序就是对话顺序
        print(f"  会话 {sid:<14} → {len(items)} 条 trace：{[i['input'] for i in items]}")
    for uid, items in by_user.items():
        # 用户视图：同一批 trace 再按 user_id 聚合，能跨会话看到「这个人一共聊了几窗」
        # {i['sessionId'] for i in items} 是集合推导，自动去重 = 会话数
        print(f"  用户 {uid:<14} → {len(items)} 条 trace，跨 "
              f"{len({i['sessionId'] for i in items})} 个会话")

In [ ]:
print_local_traces()

入口总结：三个 id = 三个粒度。密钥就绪时上面两段已经把数据上传；这段打印小结。

In [ ]:
print("\n小结：trace_id 定位「这一轮」、session_id 串起「这一窗」、user_id 归属「这个人」；")
print("      Langfuse 后台的 Sessions / Users / Traces 三个页面，就是这三种粒度的视图。")

## 3. 提示词管理（Prompt Management）

提示词**不该硬编码在代码里**（改一句要走 提交→构建→发版→回归）。Langfuse 提供提示词托管：

| 硬编码在代码里 | 托管在 Langfuse |
|---|---|
| 改一个错别字要走发版 | 界面上改完保存，下一次请求就生效 |
| 只有开发能改 | 产品 / 运营 / 提示词工程师都能改 |
| 回滚要 revert 代码 | 版本（version）天然不可变，一键切回 |
| 灰度只能靠发版切环境 | label 可以随时指向某个版本做灰度 |

三个核心概念：

| 概念 | 说明 |
|---|---|
| prompt 名称 | 提示词的唯一标识，代码里 `get_prompt("agent_system_prompt")` 用它 |
| version | 每保存一次自增 1，**内容不可变**；老版本永远拉得到 |
| label | 可移动的指针（production / staging / latest），切换「当前生效」的版本 |

变量语法是 Mustache 风格的双花括号：`你是{{user_name}}的助手。`，代码里
`prompt.compile(user_name="张三")` 把变量填进去。

### 3.1 课案原版：拉取 + 编译 + 使用

课案原版 63 行，演示最小闭环：`get_prompt` 拉托管提示词 → `compile` 填变量 →
喂给大模型。拉不到时用本地默认提示词兜底。

In [ ]:
# ---------- 课案原版：提示词管理 ----------
from langchain.chat_models import init_chat_model
from langfuse import Langfuse
from config import settings

lf = Langfuse(
    public_key=settings.langfuse_public_key,
    secret_key=settings.langfuse_secret_key,
    host=settings.langfuse_host,
)

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

In [ ]:
# ---------- 1. 拉取托管提示词 ----------
# get_prompt(名字, label/version)；label="production" 取生产版
try:
    prompt = lf.get_prompt("sql-expert", label="production")
    prompt_text = prompt.compile(variables={"dialect": "MySQL"})  # 编译变量
except Exception as e:
    # 没配置时兜底：本地默认提示词（演示用）
    print(f"未找到托管提示词（{e}），使用本地默认")
    prompt_text = "你是 MySQL 专家，把用户需求翻译成 SQL，只输出 SQL。"

# ---------- 2. 使用提示词 ----------
response = llm.invoke(
    f"{prompt_text}\n\n需求：查询每个部门的平均工资，只看平均工资大于 1 万的部门"
)
print("AI：", response.content)

# ---------- 3. （可选）代码里创建/更新提示词 ----------
# lf.create_prompt(
#     name="sql-expert",
#     prompt="你是 {dialect} 专家，把用户需求翻译成 SQL，只输出 SQL。",
#     labels=["production"],
# )
lf.flush()

### 预期输出

```text
AI： SELECT department, AVG(salary) AS avg_salary FROM employees GROUP BY department HAVING AVG(salary) > 10000;
```

> 因为本机 Langfuse 里**还没有**名为 `sql-expert` 的提示词，`get_prompt` 会走到
> `except` 分支，打印「未找到托管提示词…使用本地默认」，然后 `AI：` 输出本地默认提示词
> 驱动生成的 SQL。`AI：` 内容每次不同。

### 3.2 完整版：创建 / 拉取 / 编译 / 版本与 label

完整版 264 行，补三件事：

1. 代码里也能 `create_prompt`（适合随 CI 发布、保证多环境一致）；
2. 用 Langfuse 自己的 `Prompt_Text` 结构体在本地造提示词对象，`compile()` 行为与服务端
   拉下来完全一致，降级时也能演示；
3. 版本与 label 切换 —— 演示「线上更新不重启」的原理。

In [ ]:
# ---------- 完整版：提示词管理 ----------
import json
# create_deep_agent 只用在最后一节：证明「编译出来的提示词真的能驱动 Agent」。

from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from langfuse import Langfuse
from langfuse.api.prompts.types.prompt import Prompt_Text
from langfuse.model import TextPromptClient
from config import settings

# ---------- 0. 客户端初始化：先判断密钥是否就绪 ----------
LANGFUSE_READY = bool(settings.langfuse_public_key and settings.langfuse_secret_key)

if LANGFUSE_READY:
    langfuse = Langfuse(
        public_key=settings.langfuse_public_key,
        secret_key=settings.langfuse_secret_key,
        host=settings.langfuse_host,
    )
# 密钥为空就置 None：SDK 拿不到凭据会往 stderr 刷错误，不如干脆不实例化。
else:
    langfuse = None

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

# ---------- 1. 提示词的两种类型 ----------
# text 类型：一整段字符串，compile() 返回 str —— 适合 system prompt
PROMPT_NAME = "agent_system_prompt"
PROMPT_V1 = "你是一个乐于助人的助手。"
PROMPT_V2 = "你是{{user_name}}的专属助手，今天是{{current_date}}，回答请控制在 50 字以内。"
PROMPT_LATEST = "你是{{user_name}}的专属助手，今天是{{current_date}}。回答请控制在 50 字以内，并在结尾加一句鼓励。"
# 本地兜底：真拉不到时用它（生产代码里也建议带上 fallback，避免 Langfuse 挂了业务跟着挂）
FALLBACK_PROMPT = "你是一个助手，请用中文简洁回答。"

本地造提示词对象的降级函数 + `get_prompt` 包装。
⚠️ **label 与 version 必须二选一**：两个都传时 SDK 会在本地就抛
`ValueError: Cannot specify both version and label at the same time.`（不是服务端报错），
所以下面显式只传一个。

In [ ]:
def build_prompt_locally(text: str, version: int, labels: list[str]) -> TextPromptClient:
    """降级用：在本地造一个与服务端返回结构完全相同的 prompt 对象。

    Langfuse 从服务端拉提示词时，拿到的就是这个 Prompt_Text 结构体，
    再用它包出 TextPromptClient。所以本地这样造出来的对象，
    compile() / variables / version / labels 的行为与真的拉下来一模一样。
    """
    return TextPromptClient(Prompt_Text(
        name=PROMPT_NAME,
        version=version,
        prompt=text,
        labels=labels,
        # Prompt_Text 是 SDK 的强类型模型，字段要填全 —— 缺字段会直接报校验错误。
        tags=[],
        config={},
        commit_message=None,
    ))


def get_prompt(name: str, label: str = "production", version: int | None = None):
    """拉取提示词：真连上就查服务端，没连上就返回本地对应的版本。

    label 与 version 二选一：
        label="production"  取「当前生产版本」，服务端换 label 指向即可热更新
        version=2           取「第 2 版」这个不可变快照，用于复现历史实验
    """
    if LANGFUSE_READY:
        # fallback 参数非常关键：服务端不可用 / 这个提示词还没建，
        # SDK 会直接返回 fallback 内容而不是抛异常，保证线上不受影响
        # ⚠️ label 与 version 必须**二选一**：两个都传时 SDK 会在本地就抛
        #    `ValueError: Cannot specify both version and label at the same time.`
        #    （不是服务端报错）—— 所以这里显式只传一个，与下方降级分支保持一致。
        if version is not None:
            return langfuse.get_prompt(name, version=version, fallback=FALLBACK_PROMPT)
        return langfuse.get_prompt(name, label=label, fallback=FALLBACK_PROMPT)

    # 降级：本地三个版本，模拟服务端已有的版本历史
    local_versions = {
        1: (PROMPT_V1, ["latest"]),
        2: (PROMPT_V2, []),
        3: (PROMPT_LATEST, ["production", "latest"]),
    }
    # version 与 label 二选一：version 是不可变快照（复现历史实验），label 是可移动指针（热更新）。
    if version is not None:
        text, labels = local_versions.get(version, (FALLBACK_PROMPT, []))
        return build_prompt_locally(text, version, labels)
    if label == "production":
        return build_prompt_locally(PROMPT_LATEST, 3, ["production", "latest"])
    if label == "staging":
        return build_prompt_locally(PROMPT_V2, 2, ["staging"])
    return build_prompt_locally(PROMPT_V1, 1, ["latest"])

#### 3.2.1 一、在代码里创建提示词

In [ ]:
# ---------- 2. 在代码里创建 / 更新提示词 ----------
def demo_create_prompt():
    """课案原文是在 Web UI 上点出来；这一步演示「代码里也能建」，
    适合把提示词随 CI 流程一起发布、保证多环境一致。"""
    print("\n" + "=" * 72)
    print("一、创建提示词 create_prompt(name=..., prompt=..., labels=['production'])")
    print("=" * 72)

    payload = {
        "name": PROMPT_NAME,
        "prompt": PROMPT_LATEST,
        "labels": ["production"],       # 打上 production，代码里就能按 label 拉到它
        "type": "text",                 # text = 一整段字符串；chat = 消息数组
        "commit_message": "增加用户称呼与日期变量，限制回答长度",
    }
    if LANGFUSE_READY:
    # 真连上就调 SDK 建提示词；否则打印 create_prompt 的报文，学员照着就能在 UI 上点出来。
        prompt = langfuse.create_prompt(**payload)
        print(f"已创建：name={prompt.name} version={prompt.version} labels={prompt.labels}")
        langfuse.flush()
    else:
        print("[降级] 本应发送给 Langfuse 的 create_prompt 报文：")
        print(json.dumps(payload, ensure_ascii=False, indent=2))
        print("（Web UI 上等价操作：Prompts → New prompt → 填 name 与内容 → Save → ")
        print("  在版本列表里把 production 标签拖到想要的那一版）")

In [ ]:
demo_create_prompt()

### 预期输出

```text
========================================================================
一、创建提示词 create_prompt(name=..., prompt=..., labels=['production'])
========================================================================
已创建：name=agent_system_prompt version=<N> labels=['production', 'latest']
```

> `version=` 后面的数字**每次跑都会自增 1**（每保存一次新版本），这是 Langfuse 的
> 版本机制；`labels` 里除了你给的 `production`，还会自动带上 `latest`。

#### 3.2.2 二、拉取 + 编译 + 用于 Agent

In [ ]:
# ---------- 3. 拉取 + 编译 + 用于 Agent ----------
def demo_pull_and_compile():
    print("\n" + "=" * 72)
    print("二、拉取并编译 get_prompt(name, label='production').compile(**变量)")
    print("=" * 72)

    # 代码里写死的是 name + label，具体是哪一版由服务端决定 —— 这就是「线上更新不重启」。
    label = "production"
    prompt = get_prompt(PROMPT_NAME, label=label)
    print(f"拉取到：name={PROMPT_NAME} label={label} "
          f"version={prompt.version} labels={prompt.labels}")
    print("声明的变量：", prompt.variables)     # ['user_name', 'current_date']

    # compile 就是把 {{变量}} 填上值；缺变量会抛错，所以变量表要维护好
    system_text = prompt.compile(user_name="张三", current_date="2026-07-07")
    print("编译结果：", system_text)

    # 想直接拿去当 LangChain 的模板时，官方还提供了占位符转换（{{x}} → {x}）
    print("LangChain 模板形式：", prompt.get_langchain_prompt())

    # 课案把编译结果塞进了 DeepAgents 的 system_prompt
    agent = create_deep_agent(model=llm, tools=[], system_prompt=system_text)
    result = agent.invoke({"messages": [{"role": "user", "content": "你好"}]})
    print("AI：", result["messages"][-1].content)

    if LANGFUSE_READY:
        langfuse.flush()

In [ ]:
demo_pull_and_compile()

### 预期输出

```text
========================================================================
二、拉取并编译 get_prompt(name, label='production').compile(**变量)
========================================================================
拉取到：name=agent_system_prompt label=production version=<N> labels=['production', 'latest']
声明的变量： ['user_name', 'current_date']
编译结果： 你是张三的专属助手，今天是2026-07-07。回答请控制在 50 字以内，并在结尾加一句鼓励。
LangChain 模板形式： input_variables=['current_date', 'user_name'] ...
AI： 你好！……
```

> `version=` 数字每次不同（取决于上一格创建到第几版）；「编译结果」那一行是把
> `{{user_name}}` / `{{current_date}}` 分别填成 `张三` / `2026-07-07` 后的稳定结果。

#### 3.2.3 三、版本与 label：线上更新不重启

In [ ]:
# ---------- 4. 版本与 label：线上更新不重启 ----------
def demo_version_switch():
    """课案截图「可以添加版本，切换生产版本」讲的就是这一段。"""
    print("\n" + "=" * 72)
    print("三、版本与 label：把 production 指向新版本，代码无需改动、进程无需重启")
    print("=" * 72)

    print("课案在 Web UI 上的操作：Prompts → 选中提示词 → New version 存新内容 →")
    print("把 production 标签从 v2 拖到 v3。下面用本地版本历史模拟这个切换：\n")

    for version in (2, 3):
        p = get_prompt(PROMPT_NAME, version=version)
        print(f"  version={p.version}  labels={p.labels}")
        print(f"    原文   : {p.prompt}")
        print(f"    编译后 : {p.compile(user_name='张三', current_date='2026-07-07')}")
# 同一个 get_prompt 调用，两次拿到不同内容：变的只有服务端 label 的指向。

    print("\n  代码里写的是 label='production'，从来没改过；")
    print("  改的只是服务端那个 label 指向哪个 version —— 这就是「线上更新不重启」。")

    # 同一个 label 再拉一次：SDK 有 60 秒缓存，所以这里拿到的还是同一版
    again = get_prompt(PROMPT_NAME, label="production")
    print(f"\n  再拉一次 label='production' → version={again.version}（SDK 缓存 60s，避免每次请求都打网络）")

In [ ]:
demo_version_switch()

### 预期输出

```text
========================================================================
三、版本与 label：把 production 指向新版本，代码无需改动、进程无需重启
========================================================================
课案在 Web UI 上的操作：Prompts → 选中提示词 → New version 存新内容 →
把 production 标签从 v2 拖到 v3。下面用本地版本历史模拟这个切换：

  version=2  labels=['production', 'latest']
    原文   : 你是{{user_name}}的专属助手……
    编译后 : 你是张三的专属助手……
  version=3  labels=['production', 'latest']
    原文   : 你是{{user_name}}的专属助手……
    编译后 : 你是张三的专属助手……

  代码里写的是 label='production'，从来没改过；
  改的只是服务端那个 label 指向哪个 version —— 这就是「线上更新不重启」。

  再拉一次 label='production' → version=<N>（SDK 缓存 60s，避免每次请求都打网络）
```

> 连上真服务端时，`version=2` / `version=3` 拉的是服务端**已存在的**历史版本；
> 若服务端只有一版，可能走 fallback。降级时（密钥未就绪）则用本地三个版本精确复现。

## 小结

- **追踪**是 Langfuse 的地基：`@observe`（任意函数一行埋点）、`CallbackHandler`
  （LangChain 全自动）、`start_as_current_observation`（手工圈 span）三种方式；
- **trace / session / user 三个 id 是三种粒度**：定位「这一轮」、串起「这一窗」、
  归属「这个人」，对应后台 Traces / Sessions / Users 三个页面；
- **提示词托管**靠 `name`（写死在代码里）+ `label`（可移动指针）实现「线上更新不重启」，
  `version` 是不可变快照，`compile()` 填 Mustache 变量；
- 客户端记得 `get_client().flush()`（否则数据等批量异步刷），`get_prompt` 记得带
  `fallback`（Langfuse 挂了业务不跟着挂）。

下一课 `02_评估与打分.ipynb` 会在这批 trace 上做打分、RAG 评估与 Agent 指标评估。

## 常见坑

1. **`get_prompt(name, label=..., version=...)` 不能同时传 label 与 version**：
   SDK 会在本地就抛 `ValueError: Cannot specify both version and label at the same time.`
   （不是服务端报错）—— 二选一。第 3.2 节的 `get_prompt` 包装函数里显式只传一个。
2. **真实的 `langfuse.langchain.CallbackHandler` 没有 `.events` 属性**（只有自己写的
   「打印型」handler 才有），无条件取会 `AttributeError`。要 trace id 用
   `langfuse_handler.last_trace_id`，要当前 span 的 id 用 `get_client().get_current_trace_id()`。
3. **密钥为空时 SDK 会往 stderr 刷认证错误**（不抛异常但输出脏）：所以完整版都先算
   `LANGFUSE_READY` 再决定要不要实例化真客户端。
4. **`flush()` 是异步批量上报**，不调它数据可能等几秒才到；演示里都在关键处显式 `flush()`。
5. **`Prompt_Text` 是强类型模型，字段要填全**：缺字段会直接报校验错误（降级造对象时踩过）。
6. **`eval()` 只用于教学演示**：计算器工具里 `eval(expression)` 生产必须换成安全求值。

## 官方链接

- 可观测性总览：<https://langfuse.com/docs/observability/overview>
- 追踪（Tracing）：<https://langfuse.com/docs/tracing>
- 会话与用户：<https://langfuse.com/docs/tracing-features/sessions>
- 提示词管理：<https://langfuse.com/docs/prompt-management/get-started>
- LangChain 集成：<https://langfuse.com/docs/integrations/langchain/tracing>